In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch
import scipy.stats as stats

import gym
from gym import spaces
from PortfolioEnv2 import PortfolioEnv2

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from sklearn.model_selection import ParameterSampler

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="You provided an OpenAI Gym environment.*")

In [ ]:
#Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
data = pd.read_csv('all_sector_data-4-15.csv', parse_dates=['Date'], index_col='Date')
data.sort_index(inplace=True)

asset_cols = ['XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLK', 'XLB', 'XLRE', 'XLU']
vol_cols = ['vol20', 'vol60', 'VIX']

In [ ]:
total_rows = data.shape[0]

test_size = int(total_rows * 0.20)
trainval_size = total_rows - test_size

train_size = int(trainval_size * 0.80)
valid_size = trainval_size - train_size  

train_start = data.index[0]
train_end = data.index[train_size - 1]

validation_start = data.index[train_size]
validation_end = data.index[train_size + valid_size - 1]

test_start = data.index[train_size + valid_size]
test_end = data.index[-1]

train_data = data.loc[train_start:train_end]
valid_data = data.loc[validation_start:validation_end]
test_data  = data.loc[test_start:test_end]

print(f"train start: {train_start}, train end: {train_end}, percent of data: {train_data.shape[0]/data.shape[0]:.4f}")
print(f"validation start: {validation_start}, validation end: {validation_end}, percent of data: {valid_data.shape[0]/data.shape[0]:.4f}")
print(f"test start: {test_start}, test end: {test_end}, percent of data: {test_data.shape[0]/data.shape[0]:.4f}")

print(train_data.shape[0])
print(valid_data.shape[0])
print(test_data.shape[0])

In [ ]:
def evaluate_model(model, n_episodes=3):
    all_returns = []
    
    for _ in range(n_episodes):
        raw_eval_env = DummyVecEnv([lambda: PortfolioEnv2(valid_data, asset_cols, vol_cols, lookback=60)])
        #eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_reward=1.0)
        eval_env = VecNormalize.load("hyperparm_vec_env.pkl", raw_eval_env)
        eval_env.training    = False
        eval_env.norm_reward = False
        
        obs = eval_env.reset()
        done = False
        portfolio_values = []
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_env.step(action)
            portfolio_values.append(info[0]['portfolio_value'])

        portfolio_values = np.array(portfolio_values)
        returns = portfolio_values[1:] / portfolio_values[:-1] - 1.0
        all_returns.append(returns)

    flat_returns = np.concatenate(all_returns)
    
    mean_return = np.mean(flat_returns)
    std_return = np.std(flat_returns)

    if std_return == 0:
        sharpe_ratio = 0
    else:
        sharpe_ratio = (mean_return / std_return) * np.sqrt(252)

    return sharpe_ratio

In [ ]:
param_distributions = {
    'learning_rate': stats.loguniform(1e-5, 3e-4),
    'n_steps': [128, 256, 512, 1024, 2048],
    'batch_size': [32, 64, 128, 256],
    'gamma': stats.uniform(0.95, 0.049),
    'gae_lambda': stats.uniform(0.9, 0.08),
    'clip_range': stats.uniform(0.1, 0.2),
    'ent_coef': stats.loguniform(1e-5, 1e-2)
}

In [ ]:
n_iter_search = 200
random_params = list(ParameterSampler(param_distributions, n_iter=n_iter_search, random_state=42))
results = []

In [ ]:
best_sharpe = -np.inf
best_params = None

for i, params in enumerate(random_params):
    if params['batch_size'] > params['n_steps']:
        params['batch_size'] = params['n_steps']
    
    train_env = DummyVecEnv([lambda: PortfolioEnv2(train_data, asset_cols, vol_cols, lookback=60)])
    vec_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_reward=1.0)

    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=params['learning_rate'],
        n_steps=params['n_steps'],
        batch_size=params['batch_size'],
        gamma=params['gamma'],
        gae_lambda=params['gae_lambda'],
        clip_range=params['clip_range'],
        ent_coef=params['ent_coef'],
        verbose=0,
        seed=SEED,
        device='cpu'
    )
    vec_env.save("hyperparm_vec_env.pkl")

    model.learn(total_timesteps=25000)

    sharpe = evaluate_model(model)
    print(f"Trial {i+1}/{n_iter_search}. Sharpe Ratio: {sharpe:.4f}")

    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_params = params
print(f"Trial {i+1}/{n_iter_search}: {sharpe}")

print("\nBest Hyperparameters Found:")
print(best_params)
print(f"Best Sharpe Ratio: {best_sharpe:.4f}")

In [ ]:
sac_param_distributions = {
    "learning_rate":  stats.loguniform(1e-5, 1e-3),
    "buffer_size":    [50_000, 100_000, 200_000, 500_000],
    "batch_size":     [64, 128, 256],
    "tau":            stats.uniform(0.005, 0.05),
    "gamma":          stats.uniform(0.90, 0.10),
    "train_freq":     [1, 5, 10],
    "gradient_steps": [1, 5, 10],
    "ent_coef":       ["auto", stats.loguniform(1e-4, 1e-1)],
}

In [ ]:
n_iter_search = 100
random_params_sac = list(ParameterSampler(sac_param_distributions, n_iter=n_iter_search, random_state=42))

In [ ]:
best_sharpe_sac = -np.inf
best_params_sac = None
with open('sac_params.txt', 'a') as file:
    for i, raw_params in enumerate(random_params_sac, 1):
        params = raw_params.copy()
        
        raw_ent = params.pop("ent_coef")
        if isinstance(raw_ent, str):
            ent_coef = raw_ent              
        elif hasattr(raw_ent, "rvs"):
            ent_coef = float(raw_ent.rvs()) 
        else:
            ent_coef = float(raw_ent)       
        params["ent_coef"] = ent_coef

        train_env = DummyVecEnv([lambda: PortfolioEnv2(train_data, asset_cols, vol_cols, lookback=60)])
        vec_env   = VecNormalize(train_env, norm_obs=True, norm_reward=False, clip_reward=1.0)

        model = SAC(
            "MlpPolicy",
            vec_env,
            learning_rate = params["learning_rate"],
            buffer_size = params["buffer_size"],
            batch_size = params["batch_size"],
            tau = params["tau"],
            gamma = params["gamma"],
            train_freq = params["train_freq"],
            gradient_steps = params["gradient_steps"],
            ent_coef = ent_coef,
            verbose = 0,
            seed = SEED,
        )
        
        vec_env.save("hyperparm_vec_env.pkl")

        model.learn(total_timesteps=25_000)
        sharpe = evaluate_model(model)
        print(f"SAC Trial {i}/{n_iter_search}, Sharpe {sharpe:.4f}\n")
        print(f"{params}\n")

        file.write(f"SAC Trial {i}/{n_iter_search}, Sharpe {sharpe:.4f}\n")
        file.write(f"{params}\n")

        if sharpe > best_sharpe_sac:
            best_sharpe_sac = sharpe
            best_params_sac = params.copy()

    
    print("\Best SAC hyperparameters:")
    print(best_params_sac)
    print(f"Best SAC Sharpe Ratio: {best_sharpe_sac:.4f}")